In [ ]:
# install elan + Lean 4 toolchain
!curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | bash -s -- -y --default-toolchain leanprover/lean4:v4.11.0
import os

os.environ["PATH"] = "/root/.elan/bin:" + os.environ["PATH"]
!lean --version

In [ ]:
import os
import json
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/verify")
WORK.mkdir(exist_ok=True)

# locate harvested.jsonl from harvest-v2 kernel_sources mount
HARVEST = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "harvested.jsonl" in files:
        HARVEST = os.path.join(root, "harvested.jsonl")
        break
assert HARVEST, "harvested.jsonl not found"
print("HARVEST:", HARVEST)

# load first row with proofs
with open(HARVEST) as f:
    rows = [json.loads(l) for l in f if l.strip()]
print(f"rows: {len(rows)}")
sample = rows[0]
print(
    f"sample id={sample['id']} eq1={sample['eq1']} eq2={sample['eq2']} label={sample['label']}"
)
print(f"proofs: {len(sample['proofs'])}")
for i, p in enumerate(sample["proofs"]):
    print(f"  [{i}]: {p[:150]}")

In [ ]:
# full scan: try compile every harvested proof, save verified.jsonl
import os
import time
import re
import textwrap
import json
from pathlib import Path

PREAMBLE = """class Magma (G : Type) where
  op : G \u2192 G \u2192 G
infixl:70 " \u25c7 " => Magma.op
"""


def to_diamond(s):
    return s.replace("*", "\u25c7")


def used_vars(eqs, cand="xyzwu"):
    return "".join(v for v in cand if re.search(rf"\b{v}\b", eqs)) or "x"


def reindent(body):
    s = body.strip()
    if s.startswith("by"):
        s = s[2:].lstrip("\n")
    s = textwrap.dedent(s)
    return "by\n" + "\n".join("  " + ln if ln.strip() else "" for ln in s.split("\n"))


def build_lean(eq1, eq2, proof):
    e1, e2 = to_diamond(eq1), to_diamond(eq2)
    v = " ".join(used_vars(e1 + " " + e2))
    return (
        PREAMBLE + "theorem sair_implication\n"
        "    (G : Type) [inst : Magma G]\n"
        f"    (h : \u2200 {v} : G, {e1})\n"
        f"    : \u2200 {v} : G, {e2} := " + reindent(proof) + "\n"
    )


def check(src, timeout=15):
    p = WORK / "test.lean"
    p.write_text(src)
    try:
        r = subprocess.run(
            ["lean", str(p)], capture_output=True, text=True, timeout=timeout
        )
        return r.returncode == 0
    except subprocess.TimeoutExpired:
        return False


OUT = Path("/kaggle/working/verified.jsonl")
t0 = time.time()
total_proofs = 0
total_passed = 0
rows_with_proof = {"true": 0, "false": 0}
with OUT.open("w") as out:
    for i, r in enumerate(rows):
        verified_proofs = []
        for j, proof in enumerate(r["proofs"]):
            total_proofs += 1
            try:
                src = build_lean(r["eq1"], r["eq2"], proof)
                ok = check(src, timeout=10)
            except Exception:
                ok = False
            if ok:
                total_passed += 1
                verified_proofs.append(proof)
        if verified_proofs:
            rows_with_proof[r.get("label", "")] = (
                rows_with_proof.get(r.get("label", ""), 0) + 1
            )
        out.write(
            json.dumps(
                {
                    "id": r["id"],
                    "split": r["split"],
                    "label": r["label"],
                    "eq1": r["eq1"],
                    "eq2": r["eq2"],
                    "verified_proofs": verified_proofs,
                    "n_candidates": len(r["proofs"]),
                }
            )
            + "\n"
        )
        out.flush()
        if (i + 1) % 25 == 0 or i == 0:
            print(
                f"[{i + 1}/{len(rows)}] passed={total_passed}/{total_proofs} rows_with_proof={rows_with_proof} t={time.time() - t0:.0f}s",
                flush=True,
            )

print("=== FINAL ===", flush=True)
print(
    f"proofs passed: {total_passed}/{total_proofs} = {total_passed / total_proofs * 100:.2f}%",
    flush=True,
)
print(f"rows with at least 1 verified: {rows_with_proof}", flush=True)
print(f"time: {time.time() - t0:.0f}s", flush=True)